# Tahap 2 — Case Representation

Tujuan: representasikan setiap putusan dalam struktur data terorganisir.

Output:
- `data/processed/cases.csv` — metadata + fitur teks
- `data/processed/cases_clean.csv` — setelah cleaning lanjutan
- `data/processed/bow_features.csv` — fitur Bag-of-Words

## Import & Path

In [41]:
import os
import re
import pandas as pd
from datetime import datetime
from sklearn.feature_extraction.text import CountVectorizer

In [42]:
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

RAW_FOLDER       = os.path.join(BASE_DIR, "data", "raws")
PROCESSED_FOLDER = os.path.join(BASE_DIR, "data", "processed")

os.makedirs(PROCESSED_FOLDER, exist_ok=True)

print("Raw folder      :", RAW_FOLDER)
print("Processed folder:", PROCESSED_FOLDER)

Raw folder      : c:\Users\Rani\Downloads\CBR_Project\CBR_Project\data\raws
Processed folder: c:\Users\Rani\Downloads\CBR_Project\CBR_Project\data\processed


## Fungsi Ekstraksi Metadata

### Nomor Perkara

In [43]:
def extract_nomor_perkara(text: str):
    patterns = [
        r"p\s*u\s*t\s*u\s*s\s*a\s*n\s+nomor\s+([0-9]+)",
        r"putusan\s+nomor\s+([0-9]+)",
        r"nomor\s+([0-9]+)\s+k/pid",
        r"nomor\s+([0-9]+)"
    ]
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(1)
    return None

### Nama Terdakwa

In [44]:
def extract_terdakwa(text: str):
    match = re.search(r"nama\s*:\s*(.*?);", text, re.IGNORECASE)
    return match.group(1).strip() if match else None

### Pasal

In [45]:
def extract_pasal_utama(text: str):
    """Ekstrak pasal utama (372 / 374 untuk kasus penggelapan/pidana)."""
    pasal = re.findall(r"pasal\s+(372|374)", text, re.IGNORECASE)
    if pasal:
        return ",".join(sorted(set(pasal)))
    return None

In [46]:
def extract_pasal_lain(text: str) -> str:
    """Ekstrak semua pasal yang disebut dalam teks."""
    pasal = re.findall(r"pasal\s+(\d+)\s*([a-z\.]*)", text, re.IGNORECASE)
    hasil = []
    for nomor, aturan in pasal:
        aturan = aturan.upper().strip(".,;:")
        if aturan:
            hasil.append(f"{nomor} {aturan}")
        else:
            hasil.append(nomor)
    return ",".join(sorted(set(hasil)))

### Tanggal Putusan

In [47]:
def extract_tanggal(text: str):
    tail_text = text[int(len(text) * 0.7):]

    patterns = [
        r"rapat musyawarah majelis hakim.*?tanggal\s+(\d+\s+[a-z]+\s+\d{4})",
        r"putusan tersebut diucapkan.*?tanggal\s+(\d+\s+[a-z]+\s+\d{4})",
        r"hari\s+\w+\s*,?\s*tanggal\s+(\d+\s+[a-z]+\s+\d{4})"
    ]

    bulan = {
        "januari": "01", "februari": "02", "maret": "03",
        "april": "04",   "mei": "05",      "juni": "06",
        "juli": "07",    "agustus": "08",  "september": "09",
        "oktober": "10", "november": "11", "desember": "12"
    }

    for pattern in patterns:
        match = re.search(pattern, tail_text, re.IGNORECASE | re.DOTALL)
        if match:
            tanggal_text = match.group(1).lower()
            parts = tanggal_text.split()
            if len(parts) == 3:
                hari, nama_bulan, tahun = parts
                bulan_num = bulan.get(nama_bulan)
                if bulan_num:
                    return f"{tahun}-{bulan_num}-{int(hari):02d}"
    return None

### Ringkasan Fakta

In [48]:
def extract_ringkasan_fakta(text):

    if not text:
        return ""

    start = re.search(
        r"terdakwa\s+diajukan\s+di\s+depan\s+persidangan",
        text,
        re.IGNORECASE
    )

    if start:
        text = text[start.start():]

    end = re.search(
        r"menimbang\s*,?\s*bahwa\s+terhadap\s+alasan",
        text,
        re.IGNORECASE
    )

    if end:
        text = text[:end.start()]

    text = re.sub(r"\s+", " ", text)

    return text.strip()

### Amar Putusan

In [49]:
def extract_amar(text: str):
    pattern = (
        r"m\s*e\s*n\s*g\s*a\s*d\s*i\s*l\s*i\s*:(.*?)"
        r"demikianlah diputuskan"
    )
    match = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
    if match:
        amar = match.group(1)
        amar = re.sub(r"[•▪◦●]", " ", amar)
        amar = re.sub(r"\s+", " ", amar)
        return amar.strip()
    return None

## Bangun cases.csv

In [50]:
records = []

for file in sorted(os.listdir(RAW_FOLDER)):
    if not file.endswith(".txt"):
        continue

    path = os.path.join(RAW_FOLDER, file)
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    records.append({
        "case_id":          file.replace(".txt", ""),
        "nomor_perkara":    extract_nomor_perkara(text),
        "terdakwa":         extract_terdakwa(text),
        "pasal_utama":      extract_pasal_utama(text),
        "pasal_lain":       extract_pasal_lain(text),
        "tanggal_putusan":  extract_tanggal(text),
        "ringkasan_fakta":  extract_ringkasan_fakta(text),
        "amar_putusan":     extract_amar(text),
        "word_count":       len(text.split()),
        "text_full":        text
    })

df = pd.DataFrame(records)

print(f"Jumlah kasus: {len(df)}")
df.head()

Jumlah kasus: 33


,case_id,nomor_perkara,terdakwa,pasal_utama,pasal_lain,tanggal_putusan,ringkasan_fakta,amar_putusan,word_count,text_full
0,case_001,182,"m. umar kadavid, s.e.","372,374","197 AYAT,372 KUHP,374 KUHP,378 KUHP,488 UNDANG...",2026-03-04,terdakwa diajukan di depan persidangan pengadi...,- menolak permohonan kasasi dari pemohon kasas...,1663,p u t u s a n nomor 182 demi keadilan berdasar...
1,case_002,205,rosmawanti dewi juniarti binti (almarhum) darm...,374,"126 AYAT,244 UNDANG,253 AYAT,254 UNDANG,3 AYAT...",2026-03-04,terdakwa diajukan di depan persidangan pengadi..., mengabulkan permohonan kasasi dari pemohon k...,2246,p u t u s a n nomor 205 demi keadilan berdasar...
2,case_003,219,horas sianturi,372,"197 AYAT,253 AYAT,3 AYAT,361 HURUF,372 KITAB,3...",2026-03-05,terdakwa diajukan di depan persidangan pengadi...,− menolak permohonan kasasi dari pemohon kasas...,1991,p u t u s a n nomor 219 demi keadilan berdasar...
3,case_004,246,"darwin, s.e. anak dari ong ka beng",374,"197 AYAT,20 UNDANG,253 AYAT,374 JUNCTO,374 KUH...",2026-02-24,terdakwa diajukan di depan persidangan pengadi...,- menolak permohonan kasasi dari pemohon kasas...,2781,p u t u s a n nomor 246 demi keadilan berdasar...
4,case_005,254,sardianto bin seran,"372,374","197 AYAT,20 UNDANG,248 AYAT,253 AYAT,3 AYAT,36...",2026-02-26,terdakwa diajukan di depan persidangan pengadi...,- menyatakan tidak dapat diterima permohonan k...,2991,p u t u s a n nomor 254 demi keadilan berdasar...


In [51]:
csv_path = os.path.join(PROCESSED_FOLDER, "cases.csv")
df.to_csv(csv_path, index=False, encoding="utf-8-sig")
print("Berhasil disimpan ke:", csv_path)

Berhasil disimpan ke: c:\Users\Rani\Downloads\CBR_Project\CBR_Project\data\processed\cases.csv


In [52]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33 entries, 0 to 32
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   case_id          33 non-null     object
 1   nomor_perkara    33 non-null     object
 2   terdakwa         33 non-null     object
 3   pasal_utama      33 non-null     object
 4   pasal_lain       33 non-null     object
 5   tanggal_putusan  33 non-null     object
 6   ringkasan_fakta  33 non-null     object
 7   amar_putusan     33 non-null     object
 8   word_count       33 non-null     int64 
 9   text_full        33 non-null     object
dtypes: int64(1), object(9)
memory usage: 2.7+ KB


### Cek nilai kosong (missing)

In [53]:
print("Missing values per kolom:")
print(df.isnull().sum())

Missing values per kolom:
case_id            0
nomor_perkara      0
terdakwa           0
pasal_utama        0
pasal_lain         0
tanggal_putusan    0
ringkasan_fakta    0
amar_putusan       0
word_count         0
text_full          0
dtype: int64


## Cleaning Lanjutan pada text_full

In [54]:
def clean_text_lanjutan(text) -> str:
    """Cleaning tambahan: gabungkan huruf terpisah & normalisasi spasi."""
    if pd.isna(text):
        return ""
    text = str(text)

    text = re.sub(
        r'\b(?:[a-zA-Z]\s+){2,}[a-zA-Z]\b',
        lambda m: m.group(0).replace(" ", ""),
        text
    )

    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [55]:
df["text_full"] = df["text_full"].apply(clean_text_lanjutan)

In [56]:
output_path = os.path.join(PROCESSED_FOLDER, "cases_clean.csv")
df.to_csv(output_path, index=False, encoding="utf-8")
print("Berhasil disimpan:", output_path)

Berhasil disimpan: c:\Users\Rani\Downloads\CBR_Project\CBR_Project\data\processed\cases_clean.csv


## Feature Engineering — Bag of Words (BoW)

In [57]:
vectorizer = CountVectorizer(
    ngram_range=(1, 2),  # unigram + bigram
    min_df=2,
    max_df=0.95
)

X_bow = vectorizer.fit_transform(df["text_full"])
print("Shape BoW matrix:", X_bow.shape)

Shape BoW matrix: (33, 5055)


In [58]:
bow_df = pd.DataFrame(
    X_bow.toarray(),
    columns=vectorizer.get_feature_names_out()
)
bow_df.insert(0, "case_id", df["case_id"])

output_path = os.path.join(PROCESSED_FOLDER, "bow_features.csv")
bow_df.to_csv(output_path, index=False)
print("Berhasil disimpan ke:", output_path)

Berhasil disimpan ke: c:\Users\Rani\Downloads\CBR_Project\CBR_Project\data\processed\bow_features.csv


## Ringkasan Hasil Tahap 2

In [59]:
print("=" * 50)
print("RINGKASAN TAHAP 2 — CASE REPRESENTATION")
print("=" * 50)
print(f"Total kasus       : {len(df)}")
print(f"Kolom metadata    : {list(df.columns)}")
print(f"BoW fitur         : {X_bow.shape[1]}")
print(f"Output cases.csv  : {os.path.join(PROCESSED_FOLDER, 'cases.csv')}")
print(f"Output clean .csv : {os.path.join(PROCESSED_FOLDER, 'cases_clean.csv')}")
print(f"Output bow .csv   : {os.path.join(PROCESSED_FOLDER, 'bow_features.csv')}")

RINGKASAN TAHAP 2 — CASE REPRESENTATION
Total kasus       : 33
Kolom metadata    : ['case_id', 'nomor_perkara', 'terdakwa', 'pasal_utama', 'pasal_lain', 'tanggal_putusan', 'ringkasan_fakta', 'amar_putusan', 'word_count', 'text_full']
BoW fitur         : 5055
Output cases.csv  : c:\Users\Rani\Downloads\CBR_Project\CBR_Project\data\processed\cases.csv
Output clean .csv : c:\Users\Rani\Downloads\CBR_Project\CBR_Project\data\processed\cases_clean.csv
Output bow .csv   : c:\Users\Rani\Downloads\CBR_Project\CBR_Project\data\processed\bow_features.csv
